# Семантические признаки музыкальных треков

Чтобы искать похожую музыку, каждый трек представим акустическими признаками и эмбеддингом CLAP. Затем сравним оба способа с помощью соседей и исходных тегов MTG-Jamendo.

## 1. Окружение

Установим библиотеки, зафиксируем случайность вычислений и выберем GPU, если он доступен.

In [1]:
%pip install -q "datasets==5.1.0" "soundfile==0.14.0" "librosa==1.0.0" "numpy==2.5.4" "pandas==3.1.0" "scipy==1.18.1" "numba==0.68.0" "transformers==5.20.2" "scikit-learn==1.9.1" "umap-learn==0.5.12" "seaborn==0.13.2" "matplotlib==3.11.2" "tqdm==4.67.1"


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import platform
import random
from importlib.metadata import version

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Python:", platform.python_version())
print("Device:", DEVICE)
for package in ["torch", "datasets", "soundfile", "librosa", "numpy", "pandas", "transformers"]:
    print(f"{package}: {version(package)}")

Python: 3.12.5
Device: mps
torch: 2.14.1
datasets: 5.1.0
soundfile: 0.14.0
librosa: 1.0.0
numpy: 2.5.4
pandas: 3.1.0
transformers: 5.20.2


In [3]:
DATASET_ID = "vtsouval/mtg_jamendo_autotagging"
DATASET_REVISION = "07213140a338cdf8a8798f59c95641aedf8944a2"
MTG_METADATA_REVISION = "cafd8e20c265ed84f1e61f1c875327971f43a62f"
MTG_RAW_BASE = f"https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/{MTG_METADATA_REVISION}"
CLAP_MODEL_ID = "laion/clap-htsat-fused"
CLAP_MODEL_REVISION = "365dea6ef167def6676140ed93bbc43f84dabb28"
TARGET_TRACKS = 100
METADATA_POOL_SIZE = 4_000
MAX_TRACKS_PER_ARTIST = 2
MAX_TRACKS_PER_ALBUM = 1
CLAP_SAMPLE_RATE = 48_000
SEGMENT_SECONDS = 10
CLAP_TRACK_BATCH_SIZE = 2


## 2. Данные

Треки берутся из MTG-Jamendo. Сначала проверим структуру метаданных и декодирование аудио, чтобы ошибки источника обнаруживались до долгих вычислений.

In [4]:
from ast import literal_eval

from datasets import load_dataset

METADATA_COLUMNS = [
    "genre", "instrument", "mood_theme", "duration",
    "album_id", "artist_id", "track_id",
]

def parse_labels(value):
    labels = literal_eval(value) if isinstance(value, str) else value
    normalized = [str(label).strip().lower() for label in labels if str(label).strip()] if isinstance(labels, (list, tuple, set)) else []
    if not normalized:
        raise ValueError(f"Invalid labels: {value!r}")
    return list(dict.fromkeys(normalized))

metadata_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
    columns=METADATA_COLUMNS,
)
first_example = next(iter(metadata_stream))

missing_columns = set(METADATA_COLUMNS).difference(first_example)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if first_example["track_id"] is None or first_example["artist_id"] is None or float(first_example["duration"]) <= 0:
    raise ValueError("Invalid track metadata")
for name in ("genre", "instrument", "mood_theme"):
    parse_labels(first_example[name])

first_example

/Users/maksim/university/VK/Playlist generation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'genre': "['downtempo', 'lounge', 'easylistening', 'chillout', 'electronic']",
 'instrument': "['synthesizer']",
 'mood_theme': "['background']",
 'duration': 30,
 'album_id': 149,
 'artist_id': 87,
 'track_id': 948}

In [5]:
from io import BytesIO

import soundfile as sf
from datasets import Audio

def decode_audio(audio):
    source = BytesIO(audio["bytes"]) if audio.get("bytes") is not None else audio["path"]
    waveform, sample_rate = sf.read(source, dtype="float32", always_2d=False)
    waveform = np.asarray(waveform, dtype=np.float32)
    valid = (
        waveform.ndim == 1
        and waveform.size > 0
        and np.isfinite(waveform).all()
        and sample_rate == 16_000
        and 29 <= waveform.size / sample_rate <= 31
    )
    if not valid:
        raise ValueError("Unexpected audio format")
    return waveform, sample_rate

audio_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
).cast_column("audio", Audio(decode=False))

smoke_rows = []
for row in audio_stream.take(2):
    waveform, sample_rate = decode_audio(row["audio"])
    smoke_rows.append({
        "track_id": row["track_id"],
        "artist_id": row["artist_id"],
        "sample_rate": sample_rate,
        "duration_sec": waveform.size / sample_rate,
    })

pd.DataFrame(smoke_rows)


,track_id,artist_id,sample_rate,duration_sec
0,948,87,16000,30.062938
1,950,87,16000,30.062938


## 3. Выборка

Случайные первые строки могут содержать много треков одного автора. Поэтому метаданные перемешиваются, а выбор ограничивается двумя треками на исполнителя и одним на альбом. Для основных жанров сохраняется минимальная представленность.

In [6]:
shuffled_rows = [metadata_rows[index] for index in np.random.default_rng(SEED).permutation(len(metadata_rows))]
selected = []
selected_ids = set()
artist_counts = Counter()
album_counts = Counter()
genre_counts = Counter()

def add_row(row):
    track_id = row["track_id"]
    artist_id = row["artist_id"]
    album_id = row["album_id"]
    allowed = (
        len(selected) < TARGET_TRACKS
        and track_id not in selected_ids
        and artist_counts[artist_id] < MAX_TRACKS_PER_ARTIST
        and album_counts[album_id] < MAX_TRACKS_PER_ALBUM
    )
    if not allowed:
        return False
    selected.append(row)
    selected_ids.add(track_id)
    artist_counts[artist_id] += 1
    album_counts[album_id] += 1
    genre_counts.update(set(row["genre"]) & set(TARGET_GENRES))
    return True

for row in shuffled_rows:
    helps_quota = any(genre in row["genre"] and genre_counts[genre] < MIN_TRACKS_PER_GENRE for genre in TARGET_GENRES)
    if helps_quota:
        add_row(row)
    if all(genre_counts[genre] >= MIN_TRACKS_PER_GENRE for genre in TARGET_GENRES):
        break

for row in shuffled_rows:
    if len(selected) >= TARGET_TRACKS:
        break
    add_row(row)

missing_genres = {genre: genre_counts[genre] for genre in TARGET_GENRES if genre_counts[genre] < MIN_TRACKS_PER_GENRE}
selection_valid = (
    len(selected) == TARGET_TRACKS
    and len(selected_ids) == TARGET_TRACKS
    and max(artist_counts.values()) <= MAX_TRACKS_PER_ARTIST
    and max(album_counts.values()) <= MAX_TRACKS_PER_ALBUM
    and not missing_genres
)
if not selection_valid:
    raise RuntimeError(f"Could not build the requested sample; genre counts: {dict(genre_counts)}")

manifest_df = pd.DataFrame(selected)
print(f"Valid rows: {len(metadata_rows)}; rejected rows: {rejected_rows}")
print("Selected genres:", {genre: genre_counts[genre] for genre in TARGET_GENRES})
manifest_df.head()


NameError: name 'metadata_rows' is not defined

In [ ]:
import csv
import re
from urllib.request import urlopen

meta_url = f"{MTG_RAW_BASE}/data/raw.meta.tsv"
licenses_url = f"{MTG_RAW_BASE}/audio_licenses.txt"

official_meta = pd.read_csv(
    meta_url, sep="\t", quoting=csv.QUOTE_NONE, engine="python", on_bad_lines="skip"
)
official_meta["track_id"] = official_meta["TRACK_ID"].str.extract(r"(\d+)$", expand=False).astype(int)
official_meta = official_meta.rename(columns={
    "TRACK_NAME": "track_title",
    "ARTIST_NAME": "artist_name",
    "ALBUM_NAME": "album_name",
    "RELEASEDATE": "release_date",
    "URL": "track_url",
})[["track_id", "track_title", "artist_name", "album_name", "release_date", "track_url"]]

licenses_text = urlopen(licenses_url, timeout=60).read().decode("utf-8")
license_by_track = {}
for block in re.split(r"\r?\n\r?\n", licenses_text.strip()):
    track_match = re.search(r"(?:^|\n)\d+/(\d+)\.mp3(?:\r?$|\n)", block, flags=re.MULTILINE)
    license_match = re.search(r"https?://creativecommons\.org/licenses/\S+", block)
    if track_match and license_match:
        license_by_track[int(track_match.group(1))] = license_match.group(0)

manifest_df = manifest_df.merge(official_meta, on="track_id", how="left", validate="one_to_one")
manifest_df["license_url"] = manifest_df["track_id"].map(license_by_track)
required_metadata = ["track_title", "artist_name", "track_url", "license_url"]
if manifest_df[required_metadata].isna().any().any():
    raise ValueError("Official metadata is missing for selected tracks")

manifest_export = manifest_df.copy()
for column in ("genre", "instrument", "mood_theme"):
    manifest_export[column] = manifest_export[column].map(lambda labels: json.dumps(labels, ensure_ascii=False))
manifest_export.to_csv("playlist_manifest.csv", index=False)
manifest_df[["track_id", "track_title", "artist_name", "track_url", "license_url"]].head()


In [ ]:
from tqdm.auto import tqdm

selected_track_ids = manifest_df["track_id"].astype(int).tolist()
selected_audio_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
    columns=["track_id", "audio"],
    filters=[("track_id", "in", selected_track_ids)],
).cast_column("audio", Audio(decode=False))

audio_by_track = {
    int(row["track_id"]): decode_audio(row["audio"])
    for row in tqdm(selected_audio_stream, total=TARGET_TRACKS, desc="Loading audio")
}
missing_audio = set(selected_track_ids) - set(audio_by_track)
if missing_audio:
    raise RuntimeError(f"Audio was not loaded for {len(missing_audio)} tracks")
print(f"Loaded tracks: {len(audio_by_track)}")


## 4. Акустические признаки

Для интерпретируемого baseline из сигнала извлекаются темп, громкость, частота пересечения нуля, спектральные характеристики, хрома и MFCC.

In [ ]:
import librosa

def extract_acoustic_features(waveform, sample_rate):
    source = np.asarray(waveform)
    valid = (
        isinstance(sample_rate, (int, np.integer))
        and not isinstance(sample_rate, (bool, np.bool_))
        and sample_rate > 0
        and source.ndim == 1
        and source.size >= 2048
        and not np.issubdtype(source.dtype, np.unsignedinteger)
        and np.isfinite(source).all()
    )
    if not valid:
        raise ValueError("Expected finite mono PCM audio and a positive sample rate")
    if np.issubdtype(source.dtype, np.signedinteger):
        y = source.astype(np.float32) / max(abs(np.iinfo(source.dtype).min), np.iinfo(source.dtype).max)
    else:
        y = source.astype(np.float32)

    is_silent = bool(np.sqrt(np.mean(np.square(y), dtype=np.float64)) <= 1e-6)
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sample_rate)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sample_rate)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sample_rate, n_mfcc=13)
    onset_envelope = librosa.onset.onset_strength(y=y, sr=sample_rate)
    onset_frames = librosa.onset.onset_detect(onset_envelope=onset_envelope, sr=sample_rate, units="frames")
    tempo_reliable = bool(not is_silent and len(onset_frames) >= 3)
    tempo = float(np.asarray(librosa.feature.tempo(sr=sample_rate, onset_envelope=onset_envelope)).reshape(-1)[0]) if tempo_reliable else 0.0

    chroma_entropy = 0.0
    if not is_silent:
        chroma_profile = librosa.feature.chroma_stft(y=y, sr=sample_rate).mean(axis=1)
        if chroma_profile.sum() > np.finfo(np.float32).eps:
            probabilities = chroma_profile / chroma_profile.sum()
            positive = probabilities > 0
            chroma_entropy = float(-(probabilities[positive] * np.log(probabilities[positive])).sum() / np.log(12))

    features = {
        "tempo_bpm": tempo,
        "rms_mean": float(rms.mean()),
        "rms_std": float(rms.std()),
        "zcr_mean": float(zcr.mean()),
        "spectral_centroid_mean": float(centroid.mean()),
        "spectral_rolloff_mean": float(rolloff.mean()),
        "chroma_entropy": chroma_entropy,
        "is_silent": is_silent,
        "tempo_reliable": tempo_reliable,
    }
    for index in range(13):
        features[f"mfcc_{index + 1:02d}_mean"] = float(mfcc[index].mean())
        features[f"mfcc_{index + 1:02d}_std"] = float(mfcc[index].std())
    return features


In [ ]:
test_sr = 16_000
test_time = np.arange(test_sr * 3) / test_sr
silence = np.zeros(test_sr * 3, dtype=np.float32)
sine_440 = (0.5 * np.sin(2 * np.pi * 440 * test_time)).astype(np.float32)
click_times = np.arange(0.5, 3.0, 0.5)
click_120_bpm = librosa.clicks(times=click_times, sr=test_sr, length=test_sr * 3)

silence_features = extract_acoustic_features(silence, test_sr)
sine_features = extract_acoustic_features(sine_440, test_sr)
click_features = extract_acoustic_features(click_120_bpm, test_sr)

assert silence_features.keys() == sine_features.keys()
assert sine_features["rms_mean"] > silence_features["rms_mean"]
assert 400 < sine_features["spectral_centroid_mean"] < 500
assert not sine_features["tempo_reliable"]
assert silence_features["tempo_bpm"] == 0
assert silence_features["is_silent"]
assert not silence_features["tempo_reliable"]
assert 0 <= sine_features["chroma_entropy"] <= 1
assert click_features["tempo_reliable"]
assert min(abs(click_features["tempo_bpm"] - value) for value in (60, 120, 240)) < 5
assert len(sine_features) == 35

for bad_audio, bad_sr in [([], test_sr), ([0.0] * 2048, 0), ([np.nan] * 2048, test_sr)]:
    try:
        extract_acoustic_features(bad_audio, bad_sr)
        raise AssertionError("Expected ValueError")
    except ValueError:
        pass

pd.DataFrame([silence_features, sine_features, click_features], index=["silence", "sine_440_hz", "click_120_bpm"])

In [ ]:
acoustic_rows = []
for track_id in tqdm(selected_track_ids, desc="Extracting acoustic features"):
    waveform, sample_rate = audio_by_track[track_id]
    acoustic_rows.append({"track_id": track_id, **extract_acoustic_features(waveform, sample_rate)})

acoustic_df = pd.DataFrame(acoustic_rows)
acoustic_checks = {
    "wrong row count": len(acoustic_df) == TARGET_TRACKS,
    "duplicate track ids": acoustic_df["track_id"].is_unique,
    "non-finite feature values": np.isfinite(acoustic_df.select_dtypes(include=np.number)).all().all(),
}
acoustic_errors = [message for message, passed in acoustic_checks.items() if not passed]
if acoustic_errors:
    raise ValueError("Invalid acoustic features: " + "; ".join(acoustic_errors))
track_df = manifest_df.merge(acoustic_df, on="track_id", validate="one_to_one")
track_df[["track_id", "track_title", "tempo_bpm", "rms_mean", "spectral_centroid_mean"]].head()

## 5. CLAP

Акустические числа не описывают смысл композиции. CLAP переводит три десятисекундных фрагмента каждого трека в общее пространство аудио и текста; среднее этих векторов используется как представление всего трека.

Genre помогает находить музыку близкого стиля, mood — похожего эмоционального состояния, context — подходящую для одной ситуации, atmosphere — близкую по общему звучанию. Tempo и спектральные признаки дают простой интерпретируемый baseline, а полный CLAP-эмбеддинг сохраняет больше информации для поиска соседей.

Вместо CLAP можно использовать готовый аудиоклассификатор, но он ограничен фиксированным набором классов. LLM по названию и исполнителю не анализирует само аудио. Ручная разметка точнее для конкретной задачи, однако для 100 треков она требует отдельной проверки разметчиков.

In [ ]:
from transformers import ClapModel, ClapProcessor

clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL_ID, revision=CLAP_MODEL_REVISION)
clap_model = ClapModel.from_pretrained(CLAP_MODEL_ID, revision=CLAP_MODEL_REVISION).to(DEVICE).eval()

def make_clap_segments(waveform, sample_rate):
    target_length = CLAP_SAMPLE_RATE * SEGMENT_SECONDS * 3
    resampled = librosa.resample(np.asarray(waveform, dtype=np.float32), orig_sr=sample_rate, target_sr=CLAP_SAMPLE_RATE)
    if resampled.size < target_length:
        resampled = np.pad(resampled, (0, target_length - resampled.size))
    else:
        resampled = resampled[:target_length]
    segment_length = CLAP_SAMPLE_RATE * SEGMENT_SECONDS
    return [resampled[start:start + segment_length] for start in range(0, target_length, segment_length)]

clap_embeddings = []
for start in tqdm(range(0, TARGET_TRACKS, CLAP_TRACK_BATCH_SIZE), desc="Encoding audio"):
    batch_ids = selected_track_ids[start:start + CLAP_TRACK_BATCH_SIZE]
    segments = []
    for track_id in batch_ids:
        waveform, sample_rate = audio_by_track[track_id]
        segments.extend(make_clap_segments(waveform, sample_rate))
    inputs = clap_processor(audio=segments, sampling_rate=CLAP_SAMPLE_RATE, return_tensors="pt", padding=True)
    inputs = {name: value.to(DEVICE) if hasattr(value, "to") else value for name, value in inputs.items()}
    with torch.inference_mode():
        audio_output = clap_model.get_audio_features(**inputs)
        segment_embeddings = audio_output.pooler_output if hasattr(audio_output, "pooler_output") else audio_output
        segment_embeddings = torch.nn.functional.normalize(segment_embeddings, dim=1)
        batch_embeddings = segment_embeddings.reshape(len(batch_ids), 3, -1).mean(dim=1)
        batch_embeddings = torch.nn.functional.normalize(batch_embeddings, dim=1)
    clap_embeddings.append(batch_embeddings.cpu().numpy())

clap_embeddings = np.concatenate(clap_embeddings).astype(np.float32)
embedding_similarity = clap_embeddings @ clap_embeddings.T
off_diagonal_similarity = embedding_similarity[np.triu_indices(TARGET_TRACKS, k=1)]
embedding_checks = {
    "wrong row count": clap_embeddings.shape[0] == TARGET_TRACKS,
    "non-finite values": np.isfinite(clap_embeddings).all(),
    "vectors are not normalized": np.allclose(np.linalg.norm(clap_embeddings, axis=1), 1, atol=1e-4),
    "embeddings collapsed": np.median(off_diagonal_similarity) <= 0.95,
}
embedding_errors = [message for message, passed in embedding_checks.items() if not passed]
if embedding_errors:
    raise ValueError("Invalid CLAP embeddings: " + "; ".join(embedding_errors))
print("CLAP embedding shape:", clap_embeddings.shape)

In [ ]:
ZERO_SHOT_GROUPS = {
    "genre": ["rock", "pop", "electronic", "jazz", "classical", "folk", "hiphop", "metal", "reggae", "blues"],
    "mood": ["happy", "sad", "calm", "energetic", "romantic", "dark", "dreamy", "aggressive"],
    "context": ["work", "workout", "party", "relaxation", "study", "sleep", "driving", "meditation"],
    "atmosphere": ["atmospheric", "bright", "dark", "dreamy", "intimate", "epic", "acoustic", "electronic"],
}

PROMPT_TEMPLATES = {
    "genre": [
        "A {label} music track.",
        "Music in the {label} genre.",
        "A recording with a typical {label} sound.",
    ],
    "mood": [
        "Music with a {label} mood.",
        "A track that sounds {label}.",
        "A {label} emotional music track.",
    ],
    "context": [
        "Music suitable for {label}.",
        "A track for {label}.",
        "Background music for {label}.",
    ],
    "atmosphere": [
        "Music with a {label} atmosphere.",
        "A track with a {label} sound.",
        "A {label} musical texture.",
    ],
}

def encode_texts(texts):
    inputs = clap_processor(text=texts, return_tensors="pt", padding=True)
    inputs = {name: value.to(DEVICE) if hasattr(value, "to") else value for name, value in inputs.items()}
    with torch.inference_mode():
        output = clap_model.get_text_features(**inputs)
        embeddings = output.pooler_output if hasattr(output, "pooler_output") else output
        embeddings = torch.nn.functional.normalize(embeddings, dim=1)
    return embeddings.cpu().numpy()

def build_class_prototypes(group, labels):
    templates = PROMPT_TEMPLATES[group]
    prompts = [template.format(label=label) for label in labels for template in templates]
    prompt_embeddings = encode_texts(prompts)
    prototypes = prompt_embeddings.reshape(len(labels), len(templates), -1).mean(axis=1)
    prototypes /= np.linalg.norm(prototypes, axis=1, keepdims=True)
    return prototypes

semantic_columns = {}
for group, labels in ZERO_SHOT_GROUPS.items():
    prototypes = build_class_prototypes(group, labels)
    raw_scores = clap_embeddings @ prototypes.T
    decision_scores = raw_scores - raw_scores.mean(axis=0, keepdims=True) if group == "genre" else raw_scores
    order = np.argsort(decision_scores, axis=1)[:, ::-1]
    semantic_columns[f"{group}_pred"] = [labels[index] for index in order[:, 0]]
    semantic_columns[f"{group}_score"] = raw_scores[np.arange(TARGET_TRACKS), order[:, 0]]
    semantic_columns[f"{group}_margin"] = decision_scores[np.arange(TARGET_TRACKS), order[:, 0]] - decision_scores[np.arange(TARGET_TRACKS), order[:, 1]]

semantic_df = pd.DataFrame({"track_id": selected_track_ids, **semantic_columns})
enriched_df = track_df.merge(semantic_df, on="track_id", validate="one_to_one")
collapsed_groups = [group for group in ZERO_SHOT_GROUPS if semantic_df[f"{group}_pred"].nunique() < 2]
semantic_checks = {
    f"collapsed groups: {collapsed_groups}": not collapsed_groups,
    "wrong row count": len(enriched_df) == TARGET_TRACKS,
    "missing scores": enriched_df.filter(regex="_(score|margin)$").notna().all().all(),
}
semantic_errors = [message for message, passed in semantic_checks.items() if not passed]
if semantic_errors:
    raise ValueError("Invalid zero-shot features: " + "; ".join(semantic_errors))

enriched_export = enriched_df.copy()
for column in ("genre", "instrument", "mood_theme"):
    enriched_export[column] = enriched_export[column].map(lambda labels: json.dumps(labels, ensure_ascii=False))
enriched_export.to_csv("enriched_tracks.csv", index=False)
np.savez_compressed("clap_embeddings.npz", track_id=np.asarray(selected_track_ids), embedding=clap_embeddings)
enriched_df[["track_title", "artist_name", "genre_pred", "mood_pred", "context_pred", "atmosphere_pred"]].head()


## 6. EDA

Графики показывают состав выборки, распределения численных признаков и связь zero-shot тегов. Cosine score CLAP здесь используется как относительная мера близости, а не как вероятность.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
genre_counts_plot = enriched_df.explode("genre")["genre"].value_counts().head(15)
mood_counts_plot = enriched_df["mood_pred"].value_counts()
context_counts_plot = enriched_df["context_pred"].value_counts()
sns.barplot(x=genre_counts_plot.values, y=genre_counts_plot.index, ax=axes[0], color="steelblue")
sns.barplot(x=mood_counts_plot.values, y=mood_counts_plot.index, ax=axes[1], color="seagreen")
sns.barplot(x=context_counts_plot.values, y=context_counts_plot.index, ax=axes[2], color="darkorange")
axes[0].set_title("Reference genres")
axes[1].set_title("CLAP moods")
axes[2].set_title("CLAP contexts")
for axis in axes:
    axis.set_xlabel("Tracks")
    axis.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
reliable_tempo = enriched_df.loc[enriched_df["tempo_reliable"], "tempo_bpm"]
sns.histplot(reliable_tempo, bins=15, ax=axes[0], color="steelblue")
sns.histplot(enriched_df["rms_mean"], bins=15, ax=axes[1], color="seagreen")
sns.histplot(enriched_df["spectral_centroid_mean"], bins=15, ax=axes[2], color="darkorange")
axes[0].set_title("Reliable tempo")
axes[1].set_title("RMS energy")
axes[2].set_title("Spectral centroid")
plt.tight_layout()
plt.show()

correlation_columns = [
    "tempo_bpm", "rms_mean", "zcr_mean", "spectral_centroid_mean",
    "spectral_rolloff_mean", "chroma_entropy", "mfcc_01_mean",
]
plt.figure(figsize=(9, 7))
sns.heatmap(enriched_df[correlation_columns].corr(method="spearman"), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlation")
plt.tight_layout()
plt.show()

In [ ]:
mood_context = pd.crosstab(enriched_df["mood_pred"], enriched_df["context_pred"])
plt.figure(figsize=(10, 6))
sns.heatmap(mood_context, annot=True, fmt="d", cmap="Blues")
plt.title("Mood and context")
plt.tight_layout()
plt.show()

enriched_df["genre_hit_at_1"] = [
    predicted in reference for predicted, reference in zip(enriched_df["genre_pred"], enriched_df["genre"])
]
zero_shot_summary = pd.DataFrame({
    "metric": ["Genre hit@1", "Mean genre margin", "Mean mood margin", "Mean context margin"],
    "value": [
        enriched_df["genre_hit_at_1"].mean(),
        enriched_df["genre_margin"].mean(),
        enriched_df["mood_margin"].mean(),
        enriched_df["context_margin"].mean(),
    ],
})
zero_shot_summary

## 7. Похожие треки

Соседи сравниваются на глубинах 3, 5 и 10. Треки того же исполнителя исключаются. Релевантность определяется отдельно по жанрам, инструментам и mood/theme; частичное пересечение тегов используется как graded relevance для nDCG.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, normalize

RETRIEVAL_K = (3, 5, 10, 15)
LABEL_SPACES = ("genre", "instrument", "mood_theme")

acoustic_feature_columns = [
    column for column in acoustic_df.columns
    if column not in {"track_id", "is_silent", "tempo_reliable"}
]
acoustic_vectors = normalize(StandardScaler().fit_transform(enriched_df[acoustic_feature_columns]))
clap_similarity = cosine_similarity(clap_embeddings)
acoustic_similarity = cosine_similarity(acoustic_vectors)
random_vectors = normalize(np.random.default_rng(SEED).normal(size=(TARGET_TRACKS, 64)))
random_similarity = cosine_similarity(random_vectors)

def candidate_indices(frame, query_index):
    query_artist = frame.iloc[query_index]["artist_id"]
    return np.flatnonzero(
        (np.arange(len(frame)) != query_index)
        & frame["artist_id"].ne(query_artist).to_numpy()
    )

def nearest_neighbors(similarity, query_index, frame, k=5):
    candidates = candidate_indices(frame, query_index)
    return candidates[np.argsort(-similarity[query_index, candidates])[:k]].tolist()

def label_jaccard(left, right):
    left_set, right_set = set(left), set(right)
    union = left_set | right_set
    return len(left_set & right_set) / len(union) if union else 0.0

def ranking_metrics(retrieved_relevance, candidate_relevance):
    relevance = np.asarray(retrieved_relevance, dtype=float)
    hits = relevance > 0
    total_relevant = int(np.count_nonzero(candidate_relevance))
    ranks = np.arange(1, len(relevance) + 1)
    cumulative_hits = np.cumsum(hits)
    average_precision = float(np.sum((cumulative_hits / ranks) * hits) / min(total_relevant, len(relevance))) if total_relevant else 0.0
    reciprocal_rank = float(1 / ranks[np.flatnonzero(hits)[0]]) if hits.any() else 0.0
    discounts = 1 / np.log2(ranks + 1)
    ideal = np.sort(np.asarray(candidate_relevance, dtype=float))[::-1][:len(relevance)]
    ideal_dcg = float(np.sum(ideal * discounts))
    return {
        "hit_rate": float(hits.any()),
        "precision": float(hits.mean()),
        "recall": float(hits.sum() / total_relevant) if total_relevant else 0.0,
        "map": average_precision,
        "mrr": reciprocal_rank,
        "ndcg": float(np.sum(relevance * discounts) / ideal_dcg) if ideal_dcg else 0.0,
        "mean_jaccard": float(relevance.mean()),
    }

def evaluate_retrieval(similarity, frame, method):
    rows = []
    for query_index in range(len(frame)):
        candidates = candidate_indices(frame, query_index)
        order = np.argsort(-similarity[query_index, candidates])
        for label_space in LABEL_SPACES:
            query_labels = frame.iloc[query_index][label_space]
            relevance = np.array([label_jaccard(query_labels, frame.iloc[index][label_space]) for index in candidates])
            for k in RETRIEVAL_K:
                rows.append({
                    "method": method,
                    "label_space": label_space,
                    "k": k,
                    **ranking_metrics(relevance[order][:k], relevance),
                })
    return pd.DataFrame(rows)

similarity_spaces = {
    "CLAP": clap_similarity,
    "Acoustic": acoustic_similarity,
    "Random": random_similarity,
}
query_metrics = pd.concat([
    evaluate_retrieval(similarity, enriched_df, method)
    for method, similarity in similarity_spaces.items()
], ignore_index=True)
metric_columns = ["hit_rate", "precision", "recall", "map", "mrr", "ndcg", "mean_jaccard"]
metrics_df = query_metrics.groupby(["method", "label_space", "k"], as_index=False)[metric_columns].mean()
metrics_df.to_csv("retrieval_metrics.csv", index=False)
display(metrics_df[metrics_df["k"] == 5].pivot(index="label_space", columns="method", values="ndcg").round(3))


In [ ]:
example_rows = []
for query_index in [0, 25, 50, 75]:
    neighbors = nearest_neighbors(clap_similarity, query_index, enriched_df, k=5)
    query = enriched_df.iloc[query_index]
    for rank, neighbor_index in enumerate(neighbors, start=1):
        neighbor = enriched_df.iloc[neighbor_index]
        example_rows.append({
            "query": f"{query['artist_name']} — {query['track_title']}",
            "rank": rank,
            "neighbor": f"{neighbor['artist_name']} — {neighbor['track_title']}",
            "cosine": clap_similarity[query_index, neighbor_index],
            "query_genres": ", ".join(query["genre"]),
            "neighbor_genres": ", ".join(neighbor["genre"]),
        })

examples_df = pd.DataFrame(example_rows)
examples_df


In [ ]:
from IPython.display import Audio as NotebookAudio, display

query_index = 0
preview_indices = [query_index, *nearest_neighbors(clap_similarity, query_index, enriched_df, k=2)]
for index in preview_indices:
    row = enriched_df.iloc[index]
    waveform, sample_rate = audio_by_track[int(row["track_id"])]
    print(f"{row['artist_name']} — {row['track_title']}")
    display(NotebookAudio(waveform, rate=sample_rate))

In [ ]:
import umap

projection = umap.UMAP(
    n_neighbors=15,
    min_dist=0.15,
    metric="cosine",
    random_state=SEED,
    n_jobs=1,
).fit_transform(clap_embeddings)

def display_genre(labels):
    return next((genre for genre in TARGET_GENRES if genre in labels), "other")

plot_df = pd.DataFrame({
    "UMAP 1": projection[:, 0],
    "UMAP 2": projection[:, 1],
    "genre": enriched_df["genre"].map(display_genre),
})
plt.figure(figsize=(11, 8))
sns.scatterplot(data=plot_df, x="UMAP 1", y="UMAP 2", hue="genre", palette="tab10", s=70, alpha=0.85)
plt.title("CLAP embedding projection")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 8. Выводы

Итоговое сравнение строится по multilabel-тегам датасета. Эти теги неполные, а UMAP меняет геометрию пространства, поэтому таблица и проекция подходят для сравнения методов, но не заменяют пользовательскую оценку рекомендаций.

In [ ]:
REFERENCE_K = 10
genre_metrics = metrics_df[(metrics_df["label_space"] == "genre") & (metrics_df["k"] == REFERENCE_K)]
best_genre = genre_metrics.loc[genre_metrics["ndcg"].idxmax()]
random_genre = genre_metrics.loc[genre_metrics["method"] == "Random", "ndcg"].iloc[0]
print(f"Best genre retrieval by nDCG@{REFERENCE_K}: {best_genre['method']} ({best_genre['ndcg']:.3f})")
print(f"Random genre nDCG@{REFERENCE_K}: {random_genre:.3f}")
print(f"Tracks: {len(enriched_df)}")
print(f"Zero-shot genre hit@1: {enriched_df['genre_hit_at_1'].mean():.3f}")
margin_columns = ["genre_margin", "mood_margin", "context_margin", "atmosphere_margin"]
least_certain = enriched_df[margin_columns].mean().idxmin().removesuffix("_margin")
print(f"Least certain zero-shot group: {least_certain}")
winners = metrics_df.loc[metrics_df.groupby(["label_space", "k"])["ndcg"].idxmax(), ["label_space", "k", "method", "ndcg"]]
display(winners.sort_values(["label_space", "k"]).reset_index(drop=True).round(3))
display(metrics_df.pivot_table(index=["label_space", "k"], columns="method", values="ndcg").round(3))
